# Identity Resolution: OurAirports x Kaggle

Finds the record pairs in **OurAirports** (`id1`, left) and **Kaggle** (`id2`, right) that describe the same real-world airport, and evaluates the matchers on the gold standard `gold_standard_oa_kg.csv` (built in `create_gold_standard.ipynb`).

**Matching does not use `icao` / `iata`.** The gold standard labels are derived from these codes, so using them as features would leak the label (see also `docs/idea_abstract.md`). Matching uses name, coordinates, city, country and the OurAirports record type.

**Steps**

1. Load the data and fix two known data issues (Kaggle `(0, 0)` coordinates, Namibia's country code `NA`)
2. Blocking: three rules (name token, geographic grid, city token), evaluated with pair completeness and reduction ratio
3. Comparators
4. Helpers: one-to-one matching, evaluation, and a precision check outside the gold standard (Kaggle railway / bus stations)
5. Rule-based matcher: weighted sum, threshold chosen on the train/validation split
6. ML matchers: trained on the gold pairs plus *implied negatives*; logistic regression, random forest and gradient boosting compared with 5-fold cross-validation
7. Final evaluation on the test split (used only here)
8. Error analysis
9. Save the correspondences for data fusion

**Gold standard:** `gold_standard_oa_kg.csv` with seven groups (hard / normal / code-changed / IATA positives, hard / normal / heliport negatives), 30 % test, the rest split into 5 folds. All choices (blocking settings, weights, thresholds, model) are made on the train/validation split; the test split is evaluated once at the end.

The `iata positive` group (Kaggle records with an IATA code but no ICAO code) contains 119 Kaggle records without coordinates. It is the only group that tests records without coordinates; there is no negative group for them yet, so precision on these records is not measured by the gold standard.

In [1]:
import math
from collections import Counter
from pathlib import Path

import h3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import textdistance
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from PyDI.entitymatching import (
    BaseComparator,
    EntityMatchingEvaluator,
    FeatureExtractor,
    GreedyOneToOneMatchingAlgorithm,
    RuleBasedMatcher,
    TokenBlocker,
)

SEED = 1234

## 1. Load the data

Two fixes are applied to the normalized data:
- **Kaggle `(0, 0)` coordinates** are missing values, not a real location (319 records). They are set to `NaN`, the same as in `create_gold_standard.ipynb`.
- **Namibia's country code is `NA`**, which pandas reads as a missing value by default. Both files are read with `keep_default_na=False`. In `kaggleNormalized.csv` the code is already missing for the Namibian airports, so it is restored from `countryCodeA3 = NAM`. (Ideally this would be fixed in the Kaggle normalization.)

In [2]:
DATA_DIR = Path("../../../data/normalized_outputs")
GOLD_FILE = Path("../../../data/identity_resolution/gold_standard/gold_standard_oa_kg.csv")
OUTPUT_DIR = Path("../../../data/identity_resolution/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

read = dict(keep_default_na=False, na_values=[""], low_memory=False)
df_oa = pd.read_csv(DATA_DIR / "ourairportsNormalized.csv", **read)
df_kg = pd.read_csv(DATA_DIR / "kaggleNormalized.csv", **read)
df_oa.attrs["source"] = "ourairports"
df_kg.attrs["source"] = "kaggle"

zero_coord = (df_kg["lat"] == 0.0) & (df_kg["lon"] == 0.0)
df_kg.loc[zero_coord, ["lat", "lon"]] = np.nan
print(f"Kaggle: set {int(zero_coord.sum())} (0, 0) coordinates to NaN")

namibia = df_kg["countryCode"].isna() & (df_kg["countryCodeA3"] == "NAM")
df_kg.loc[namibia, "countryCode"] = "NA"
print(f"Kaggle: restored countryCode 'NA' for {int(namibia.sum())} Namibian airports")
print(f"OurAirports: {int((df_oa['countryCode'] == 'NA').sum())} Namibian airports keep countryCode 'NA'")

print(f"\nOurAirports: {len(df_oa)} records | Kaggle: {len(df_kg)} records")

Kaggle: set 319 (0, 0) coordinates to NaN
Kaggle: restored countryCode 'NA' for 14 Namibian airports
OurAirports: 303 Namibian airports keep countryCode 'NA'

OurAirports: 86151 records | Kaggle: 6391 records


In [3]:
gold = pd.read_csv(GOLD_FILE)
gold_train_val = gold[gold["split"] == "train_val"].reset_index(drop=True)
gold_test = gold[gold["split"] == "test"].reset_index(drop=True)

print(gold.groupby(["split", "label"]).size().unstack().rename(columns={0: "negatives", 1: "positives"}))
print()
print(gold["type"].value_counts())

label      negatives  positives
split                          
test             170        264
train_val        396        617

type
hard positive            250
normal positive          250
iata positive            250
hard negative            250
normal negative          250
code changed positive    131
heliport negative         66
Name: count, dtype: int64


## 2. Blocking

Comparing every OurAirports record with every Kaggle record would mean 86 151 x 6 391 ≈ 550 million pairs. Three blocking rules generate the candidate pairs, and their union is matched:

- **Rule A: `countryCode` + distinguishing `name` token** (as in `create_gold_standard.ipynb`). A token is excluded if it would create more than `MAX_PAIRS_PER_TOKEN` pairs (number of OurAirports names containing it x number of Kaggle names containing it).
- **Rule B: geographic grid (H3)**, each record emits its cell plus the 6 neighbouring cells. Resolution **7** instead of 9: the same airport has different coordinates in the two sources (the median distance of the true matches is ~0.2 km, but 10 % are more than 12 km apart), and at resolution 9 the 7 cells cover only ~0.5 km.
- **Rule C: `countryCode` + `city` token.** Catches true matches with very different names and coordinates that are too far apart for Rule B (e.g. `ternopil international airport` / `ternopol`). City tokens are excluded with the same pair-count rule.

**Pair completeness** = share of the gold positives that are candidate pairs. A pair that blocking misses can never be matched.

In [4]:
def name_tokens(text):
    if not isinstance(text, str):
        return set()
    return {t for t in text.split() if len(t) >= 2}


def city_tokens(text):
    if not isinstance(text, str):
        return set()
    return {t for t in text.split() if len(t) >= 3}


def canonical(pairs):
    return pairs[["id1", "id2"]].drop_duplicates().reset_index(drop=True)


gold_positive_set = set(zip(gold.loc[gold["label"] == 1, "id1"], gold.loc[gold["label"] == 1, "id2"]))


def pair_completeness(pairs):
    found = set(zip(pairs["id1"], pairs["id2"]))
    return sum(p in found for p in gold_positive_set) / len(gold_positive_set)

### Rule A: `countryCode` + distinguishing `name` token

In [5]:
MAX_PAIRS_PER_TOKEN = 2000

doc_freq_oa = Counter(t for n in df_oa["name"] for t in name_tokens(n))
doc_freq_kg = Counter(t for n in df_kg["name"] for t in name_tokens(n))
COMMON_NAME_TOKENS = {t for t, c in doc_freq_oa.items() if c * doc_freq_kg.get(t, 0) > MAX_PAIRS_PER_TOKEN}
print(f"Excluding {len(COMMON_NAME_TOKENS)} name tokens, e.g. {sorted(COMMON_NAME_TOKENS)[:12]}")

for d in (df_oa, df_kg):
    d["name_with_country"] = d["countryCode"].fillna("") + "|" + d["name"].fillna("")


def country_name_tokenizer(text):
    country, _, name = text.partition("|")
    if not country:
        return []
    return [f"{country}_{t}" for t in name_tokens(name) if t not in COMMON_NAME_TOKENS]


blocker_a = TokenBlocker(df_left=df_oa, df_right=df_kg, column="name_with_country",
                         id_column="id", tokenizer=country_name_tokenizer)
candidates_a = canonical(blocker_a.materialize())
print(f"Rule A: {len(candidates_a)} candidate pairs, pair completeness {pair_completeness(candidates_a):.3f}")

Excluding 53 name tokens, e.g. ['aerodrome', 'air', 'airfield', 'airpark', 'airport', 'airstrip', 'army', 'base', 'bay', 'campo', 'center', 'central']


Rule A: 41458 candidate pairs, pair completeness 0.793


### Rule B: geographic grid (H3, resolution 7)

In [6]:
RESOLUTION = 7

grid_token_lookup = {}
for d in (df_oa, df_kg):
    for record_id, lat, lon in zip(d["id"], d["lat"], d["lon"]):
        if pd.isna(lat) or pd.isna(lon):
            grid_token_lookup[record_id] = []
        else:
            grid_token_lookup[record_id] = list(h3.grid_disk(h3.latlng_to_cell(lat, lon, RESOLUTION), 1))

for d in (df_oa, df_kg):
    d["id_as_text"] = d["id"]

blocker_b = TokenBlocker(df_left=df_oa, df_right=df_kg, column="id_as_text",
                         id_column="id", tokenizer=lambda record_id: grid_token_lookup.get(record_id, []))
candidates_b = canonical(blocker_b.materialize())
print(f"Rule B: {len(candidates_b)} candidate pairs, pair completeness {pair_completeness(candidates_b):.3f}")

Rule B: 9829 candidate pairs, pair completeness 0.743


### Rule C: `countryCode` + `city` token

In [7]:
MAX_PAIRS_PER_CITY_TOKEN = 500

city_freq_oa = Counter((c, t) for c, n in zip(df_oa["countryCode"], df_oa["city"]) for t in city_tokens(n))
city_freq_kg = Counter((c, t) for c, n in zip(df_kg["countryCode"], df_kg["city"]) for t in city_tokens(n))
COMMON_CITY_TOKENS = {k for k, c in city_freq_oa.items() if c * city_freq_kg.get(k, 0) > MAX_PAIRS_PER_CITY_TOKEN}
print(f"Excluding {len(COMMON_CITY_TOKENS)} (country, city token) keys")

for d in (df_oa, df_kg):
    d["city_with_country"] = d["countryCode"].fillna("") + "|" + d["city"].fillna("")


def country_city_tokenizer(text):
    country, _, city = text.partition("|")
    if not country:
        return []
    return [f"{country}_{t}" for t in city_tokens(city) if (country, t) not in COMMON_CITY_TOKENS]


blocker_c = TokenBlocker(df_left=df_oa, df_right=df_kg, column="city_with_country",
                         id_column="id", tokenizer=country_city_tokenizer)
candidates_c = canonical(blocker_c.materialize())
print(f"Rule C: {len(candidates_c)} candidate pairs, pair completeness {pair_completeness(candidates_c):.3f}")

Excluding 45 (country, city token) keys


Rule C: 48175 candidate pairs, pair completeness 0.742


### Union and blocking evaluation

In [8]:
candidates = canonical(pd.concat([candidates_a, candidates_b, candidates_c]))

blocking_eval = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=candidates, test_pairs=gold, blocker=blocker_a,
)
print(f"Candidate pairs:      {blocking_eval['total_candidates']:,}")
print(f"Possible pairs:       {blocking_eval['total_possible_pairs']:,}")
print(f"Reduction ratio:      {blocking_eval['reduction_ratio']:.5f}")
print(f"Pair completeness:    {blocking_eval['pair_completeness']:.3f} "
      f"({blocking_eval['true_positives_found']} of {blocking_eval['total_true_pairs']} gold positives)")

rows = []
for label, pairs in [("A", candidates_a), ("B", candidates_b), ("C", candidates_c),
                     ("A + B", pd.concat([candidates_a, candidates_b])),
                     ("A + B + C", candidates)]:
    rows.append({"rules": label, "pairs": len(canonical(pairs)), "pair_completeness": round(pair_completeness(pairs), 3)})
pd.DataFrame(rows)

Candidate pairs:      82,004
Possible pairs:       550,591,041
Reduction ratio:      0.99985
Pair completeness:    0.978 (862 of 881 gold positives)


,rules,pairs,pair_completeness
0,A,41458,0.793
1,B,9829,0.743
2,C,48175,0.742
3,A + B,45555,0.960
4,A + B + C,82004,0.978


**Gold positives missed by blocking.** These pairs can never be matched. The table shows why: very different names and cities, coordinates far apart, or a different `countryCode`. Most of the `iata positive` pairs here have **no Kaggle coordinates**, so Rule B cannot find them, and their names are spelled differently in the two sources (`kokhanok` / `kakhonak`, `dibba` / `dibaa`, `myrhorod` / `mirgorod`), so Rules A and C, which need an exact shared token, cannot find them either.

In [9]:
candidate_set = set(zip(candidates["id1"], candidates["id2"]))
oa_idx = df_oa.set_index("id")
kg_idx = df_kg.set_index("id")


def haversine_km(lat1, lon1, lat2, lon2):
    if any(pd.isna(v) for v in (lat1, lon1, lat2, lon2)):
        return np.nan
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    a = (math.sin(math.radians(lat2 - lat1) / 2) ** 2
         + math.cos(phi1) * math.cos(phi2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2)
    return 2 * 6371.0 * math.asin(math.sqrt(a))


def describe_pairs(pairs):
    out = pairs.copy()
    for col in ["name", "city", "countryCode"]:
        out[f"{col}_oa"] = out["id1"].map(oa_idx[col])
        out[f"{col}_kg"] = out["id2"].map(kg_idx[col])
    out["distance_km"] = [
        round(haversine_km(oa_idx.at[a, "lat"], oa_idx.at[a, "lon"], kg_idx.at[b, "lat"], kg_idx.at[b, "lon"]), 1)
        for a, b in zip(out["id1"], out["id2"])
    ]
    return out


missed = gold[(gold["label"] == 1) & ~pd.Series(list(zip(gold["id1"], gold["id2"]))).isin(candidate_set)]
print(f"{len(missed)} gold positives are not candidate pairs")
describe_pairs(missed)[["type", "name_oa", "name_kg", "city_oa", "city_kg", "countryCode_oa", "countryCode_kg", "distance_km"]]

19 gold positives are not candidate pairs


,type,name_oa,name_kg,city_oa,city_kg,countryCode_oa,countryCode_kg,distance_km
7,hard positive,jieyang chaoshan international airport,wai sha airport,jieyang rongcheng,shantou,CN,CN,24.9
40,hard positive,sazena airfield,sazena,sazena,lakeside texas,CZ,US,16.5
63,hard positive,cherkasy international airport,cherkassy,cherkasy,cherkasi,UA,UA,10.0
101,hard positive,los colonizadores airport,los colonizadores airport,saravena,farafangana,CO,MG,6.1
655,iata positive,gahcho kue,gush katif,kennedy lake,gush katif,CA,IL,NaN
665,iata positive,kualanamu international airport,kuala namu international airport,beringin,medan,ID,ID,NaN
666,iata positive,danilo atienza air base,sangley point naval air station,cavite,manila,PH,PH,NaN
668,iata positive,kimaam airport,kimam,kimaam,kimam,ID,ID,NaN
670,iata positive,naukati bay seaplane base,naukiti,tuxekan island,naukiti alaska,US,US,NaN
673,iata positive,myrhorod air base,mirgorod,myrhorod,mirgorod,UA,UA,NaN


## 3. Comparators

`icao` and `iata` are deliberately not used (label leakage, see top).

| Comparator | Compares | Why |
|---|---|---|
| `name_jw` | Jaro-Winkler of the full names | General name similarity |
| `name_core_jw` | Jaro-Winkler of the names without generic words (`airport`, `international`, `field`, ...) | Kaggle names often lack `airport` (`kapit` / `kapit airport`) |
| `name_core_overlap` | Overlap coefficient of the core name tokens: shared tokens / size of the smaller set | `pilanesberg` is fully contained in `pilanesberg international airport` |
| `city_overlap` | Overlap coefficient of the city tokens | Kaggle `city` often contains the state: `clovis new mexico` vs `clovis` |
| `kaggle_city_in_oa` | Share of the Kaggle city tokens found in the OurAirports name or city | Kaggle `city` is often only in the OurAirports name (`san jose` / `san jose juan santamaria intl`) |
| `distance` | Haversine distance between the two locations | Strongest signal; Kaggle coordinates are missing for 319 records |
| `country_equal` | 1 if `countryCode` is equal | |
| `oa_type` | OurAirports record type as a number (large 1.0 ... closed 0.1) | Kaggle lists mainly airports with scheduled flights |

The comparators return `NaN` when a value is missing; the gradient boosting model handles missing values directly, the other models impute the median. For the rule-based matcher, missing values get a fixed value instead (see section 4).

`scheduledService` was also tried as a feature during development and dropped: it did not improve the cross-validated F1 score of the gradient boosting model (0.970 without it vs 0.967 with it).

In [10]:
GENERIC_NAME_WORDS = {
    "airport", "international", "airfield", "aerodrome", "airstrip", "regional", "municipal", "field",
    "air", "base", "airbase", "force", "army", "naval", "domestic", "national", "county", "airpark",
    "heliport", "station", "city", "the", "aeroporto", "aeropuerto", "aeroport", "aerodromo",
}
OA_TYPE_SCORE = {
    "large_airport": 1.0, "medium_airport": 0.8, "small_airport": 0.5, "seaplane_base": 0.3,
    "heliport": 0.2, "balloonport": 0.1, "closed": 0.1,
}
JARO_WINKLER = textdistance.jaro_winkler.normalized_similarity


def tokens(text):
    return text.split() if isinstance(text, str) else []


def core_tokens(text):
    return [t for t in tokens(text) if t not in GENERIC_NAME_WORDS]


def overlap_coefficient(a, b):
    a, b = set(a), set(b)
    if not a or not b:
        return np.nan
    return len(a & b) / min(len(a), len(b))


class FunctionComparator(BaseComparator):
    # Wraps a function f(record_oa, record_kg) -> float, so it can be used by PyDI matchers

    def __init__(self, name, function, missing=np.nan):
        super().__init__(name=name)
        self.function = function
        self.missing = missing

    def compare(self, record1, record2):
        value = self.function(record1, record2)
        return self.missing if pd.isna(value) else float(value)


def name_jw(r1, r2):
    a, b = r1.get("name"), r2.get("name")
    return JARO_WINKLER(a, b) if isinstance(a, str) and isinstance(b, str) else np.nan


def name_core_jw(r1, r2):
    a, b = core_tokens(r1.get("name")), core_tokens(r2.get("name"))
    return JARO_WINKLER(" ".join(a), " ".join(b)) if a and b else np.nan


def name_core_overlap(r1, r2):
    return overlap_coefficient(core_tokens(r1.get("name")), core_tokens(r2.get("name")))


def city_overlap(r1, r2):
    return overlap_coefficient(tokens(r1.get("city")), tokens(r2.get("city")))


def kaggle_city_in_oa(r1, r2):
    kaggle_city = set(tokens(r2.get("city")))
    if not kaggle_city:
        return np.nan
    oa_text = set(tokens(r1.get("name")) + tokens(r1.get("city")))
    return len(kaggle_city & oa_text) / len(kaggle_city)


def distance_km(r1, r2):
    return haversine_km(r1.get("lat"), r1.get("lon"), r2.get("lat"), r2.get("lon"))


def country_equal(r1, r2):
    a, b = r1.get("countryCode"), r2.get("countryCode")
    return np.nan if pd.isna(a) or pd.isna(b) else float(a == b)


def oa_type(r1, r2):
    return OA_TYPE_SCORE.get(r1.get("type"), np.nan)


FEATURE_FUNCTIONS = {
    "name_jw": name_jw,
    "name_core_jw": name_core_jw,
    "name_core_overlap": name_core_overlap,
    "city_overlap": city_overlap,
    "kaggle_city_in_oa": kaggle_city_in_oa,
    "log_distance_km": lambda r1, r2: np.log1p(distance_km(r1, r2)),
    "country_equal": country_equal,
    "oa_type": oa_type,
}
feature_comparators = [FunctionComparator(name, fn) for name, fn in FEATURE_FUNCTIONS.items()]

# Sanity check on one gold positive and one hard negative
for _, pair in pd.concat([gold[gold["type"] == "normal positive"].head(1),
                          gold[gold["type"] == "hard negative"].head(1)]).iterrows():
    r1, r2 = oa_idx.loc[pair["id1"]], kg_idx.loc[pair["id2"]]
    print(f"{pair['type']}: {r1['name']!r} vs {r2['name']!r}")
    for comp in feature_comparators:
        print(f"  {comp.name:<18} -> {comp.compare(r1, r2):.3f}")

normal positive: 'abraham lincoln capital airport' vs 'abraham lincoln capital'
  name_jw            -> 0.948
  name_core_jw       -> 1.000
  name_core_overlap  -> 1.000
  city_overlap       -> 1.000
  kaggle_city_in_oa  -> 0.500
  log_distance_km    -> 0.001
  country_equal      -> 1.000
  oa_type            -> 0.800
hard negative: 'capital city airport' vs 'capital city airport'
  name_jw            -> 1.000
  name_core_jw       -> 1.000
  name_core_overlap  -> 1.000
  city_overlap       -> 0.000
  kaggle_city_in_oa  -> 0.000
  log_distance_km    -> 6.594
  country_equal      -> 1.000
  oa_type            -> 0.500


## 4. Helpers: one-to-one matching and evaluation

**One-to-one matching.** A Kaggle airport matches at most one OurAirports airport and vice versa, but a matcher scores every candidate pair independently, so one Kaggle airport often gets several high scores (the airport itself, a helipad in the same city, a nearby air base). `GreedyOneToOneMatchingAlgorithm` sorts the correspondences by score and keeps a pair only if neither record is already matched.

`MaximumBipartiteMatching` is **not** used: in the installed PyDI version it calls `networkx.bipartite.maximum_matching`, which maximizes the *number* of pairs and ignores the scores, so it drops high-scoring true matches in favour of low-scoring ones (with it, the test recall of the ML matcher fell from 0.96 to 0.68 in a first run).

**Evaluation.** Two workarounds for the installed PyDI version:
- `EntityMatchingEvaluator.threshold_sweep` calls a method `evaluate` that does not exist and returns an empty table, so the sweep calls `evaluate_matching` once per threshold.
- `evaluate_matching` raises `IndexError` if the gold pairs do not have a `0..n-1` index (e.g. one cross-validation fold), so the index is reset first.

Gold pairs missed by blocking count as false negatives.

In [11]:
def one_to_one(correspondences, threshold):
    return GreedyOneToOneMatchingAlgorithm(threshold=threshold).cluster(correspondences)


def evaluate(correspondences, test_pairs, threshold):
    # evaluate_matching mixes up row positions and index labels, so test_pairs needs a 0..n-1 index
    test_pairs = test_pairs[["id1", "id2", "label"]].reset_index(drop=True)
    result = EntityMatchingEvaluator.evaluate_matching(correspondences, test_pairs, threshold=threshold)
    return {k: result[k] for k in ["precision", "recall", "f1", "true_positives", "false_positives", "false_negatives"]}


def threshold_sweep(correspondences, test_pairs, thresholds, with_one_to_one=True):
    rows = []
    for t in thresholds:
        matched = one_to_one(correspondences, t) if with_one_to_one else correspondences
        rows.append({"threshold": t, **evaluate(matched, test_pairs, t),
                     "matches_in_dataset": int((matched["score"] >= t).sum())})
    return pd.DataFrame(rows)

### Precision check outside the gold standard: Kaggle records that are not airports

The gold standard has a blind spot: its negatives are pairs of two airports with an ICAO code, so it never tests whether the matcher wrongly matches a Kaggle record to a helipad or closed airfield, or a Kaggle record that has no counterpart at all. Kaggle contains railway, bus and ferry stations (they have IATA codes), and OurAirports has no such records, so **every match of a station is wrong**. The number of matched stations is used as a second precision check next to the gold standard.

In [12]:
STATION_WORDS = r"\b(?:railway|rail|train|bus|hbf|hauptbahnhof|bahnhof|ferry|station)\b"
AIRPORT_WORDS = r"\b(?:air|airport|heliport|airfield)\b"
stations = set(df_kg.loc[df_kg["name"].str.contains(STATION_WORDS, na=False)
                         & ~df_kg["name"].str.contains(AIRPORT_WORDS, na=False), "id"])
print(f"{len(stations)} Kaggle records are railway / bus / ferry stations, e.g.:")
print(df_kg.loc[df_kg["id"].isin(stations), "name"].head(6).tolist())


def matched_stations(matched):
    return int(matched["id2"].isin(stations).sum())

257 Kaggle records are railway / bus / ferry stations, e.g.:
['tokyo bus station', 'vologda rail station', 'pskov rail station', 'bern railway station', 'centrale railway station', 'porta garibaldi station']


## 5. Rule-based matcher

Weighted sum of four similarities. During development, four weight sets were compared with 5-fold cross-validation on the train/validation split (CV F1 between 0.884 and 0.905); the best set is used.

| Similarity | Weight | Definition | Missing value |
|---|---|---|---|
| name | 0.30 | max(`name_jw`, `name_core_jw`) | 0 |
| distance | 0.40 | exp(−distance / 5 km): 1 at 0 km, 0.37 at 5 km, 0.02 at 20 km | 0.5 (neutral) |
| city | 0.20 | max(`city_overlap`, `kaggle_city_in_oa`) | 0.5 (neutral) |
| country | 0.10 | `country_equal` | 0 |

Distance gets the largest weight because hard negatives have names as similar as true matches (e.g. two different `capital city airport`s), but lie ~20 km apart.

In [13]:
def rule_name(r1, r2):
    values = [v for v in (name_jw(r1, r2), name_core_jw(r1, r2)) if not pd.isna(v)]
    return max(values) if values else np.nan


def rule_distance(r1, r2):
    return math.exp(-distance_km(r1, r2) / 5.0)


def rule_city(r1, r2):
    values = [v for v in (city_overlap(r1, r2), kaggle_city_in_oa(r1, r2)) if not pd.isna(v)]
    return max(values) if values else np.nan


rule_comparators = [
    FunctionComparator("name", rule_name, missing=0.0),
    FunctionComparator("distance", rule_distance, missing=0.5),
    FunctionComparator("city", rule_city, missing=0.5),
    FunctionComparator("country", country_equal, missing=0.0),
]
rule_weights = [0.30, 0.40, 0.20, 0.10]

rule_correspondences = RuleBasedMatcher().match(
    df_left=df_oa, df_right=df_kg, candidates=candidates, id_column="id",
    comparators=rule_comparators, weights=rule_weights, threshold=0.0,
)
print(f"Scored {len(rule_correspondences)} candidate pairs")

Scored 82004 candidate pairs


### Choose the threshold on the train/validation split (after one-to-one matching)

In [14]:
rule_sweep = threshold_sweep(rule_correspondences, gold_train_val, [round(t, 2) for t in np.arange(0.40, 0.81, 0.04)])
rule_sweep["matched_stations"] = [matched_stations(one_to_one(rule_correspondences, t)) for t in rule_sweep["threshold"]]
RULE_THRESHOLD = float(rule_sweep.loc[rule_sweep["f1"].idxmax(), "threshold"])
print(f"Best threshold on train/validation: {RULE_THRESHOLD:.2f}")
rule_sweep[["threshold", "precision", "recall", "f1", "matches_in_dataset", "matched_stations"]].round(3)

Best threshold on train/validation: 0.40


,threshold,precision,recall,f1,matches_in_dataset,matched_stations
0,0.40,0.997,0.945,0.970,6284,211
1,0.44,0.997,0.943,0.969,6271,208
2,0.48,0.997,0.932,0.963,6237,199
3,0.52,0.997,0.929,0.961,6188,183
4,0.56,0.997,0.927,0.961,6155,165
5,0.60,0.996,0.906,0.949,6093,156
6,0.64,0.998,0.849,0.918,5929,141
7,0.68,0.998,0.817,0.898,5776,102
8,0.72,0.998,0.776,0.873,5587,59
9,0.76,0.998,0.739,0.849,5447,45


## 6. ML matchers

### Features
The eight comparators of section 3, computed with PyDI's `FeatureExtractor` once for all candidate pairs and once for the gold pairs (a few gold pairs are not candidates).

In [15]:
feature_extractor = FeatureExtractor(feature_comparators)
FEATURES = feature_extractor.get_feature_names()

candidate_features = feature_extractor.create_features(df_oa, df_kg, candidates, id_column="id")
gold_features = feature_extractor.create_features(df_oa, df_kg, gold[["id1", "id2"]], id_column="id")
gold_features = gold_features.merge(gold, on=["id1", "id2"])
print(f"Candidate feature matrix: {candidate_features.shape} | gold feature matrix: {gold_features.shape}")
gold_features.groupby("type")[FEATURES].median().round(2)

Candidate feature matrix: (82004, 10) | gold feature matrix: (1447, 15)


,name_jw,name_core_jw,name_core_overlap,city_overlap,kaggle_city_in_oa,log_distance_km,country_equal,oa_type
type,,,,,,,,
code changed positive,0.89,1.00,1.0,1.00,1.0,0.09,1.0,0.8
hard negative,0.89,0.89,1.0,0.67,0.5,3.07,1.0,0.8
hard positive,0.84,0.89,1.0,1.00,1.0,1.19,1.0,0.8
heliport negative,0.56,0.56,0.0,1.00,1.0,1.84,1.0,0.2
iata positive,0.89,1.00,1.0,1.00,1.0,0.21,1.0,0.5
normal negative,0.68,0.68,0.5,0.00,0.0,6.90,1.0,0.8
normal positive,0.92,1.00,1.0,1.00,1.0,0.01,1.0,0.8


### Training data: gold pairs plus implied negatives

The gold negatives are pairs of two airports with an ICAO code, so they contain **no helipads, closed airfields or air bases in the same city**. A model trained only on them has never seen such a pair and gives many of them a high probability: in a first run, 20 531 candidate pairs (for 6 391 Kaggle records) got a probability ≥ 0.5, e.g. Kaggle `franca airport` / `edificio prime business center helipad` with 0.999.

**Implied negatives** fill this gap: if Kaggle record X matches OurAirports record Y, then X matches no other OurAirports record and Y no other Kaggle record. So every other candidate pair of X or Y is a negative. They are taken only from the positives of the training data, and pairs that involve any record of the test split are left out, so no information from the test split is used.

Limitation: if OurAirports contains a duplicate of Y (see the OurAirports local matching), the pair with the duplicate becomes a wrong negative. This affects few pairs.

In [16]:
test_ids = set(gold_test["id1"]) | set(gold_test["id2"])


def implied_negatives(positives, exclude_ids):
    kaggle_to_oa = dict(zip(positives["id2"], positives["id1"]))
    oa_to_kaggle = dict(zip(positives["id1"], positives["id2"]))
    c = candidate_features
    other_oa = c["id2"].isin(kaggle_to_oa.keys()) & (c["id2"].map(kaggle_to_oa) != c["id1"])
    other_kaggle = c["id1"].isin(oa_to_kaggle.keys()) & (c["id1"].map(oa_to_kaggle) != c["id2"])
    keep = (other_oa | other_kaggle) & ~c["id1"].isin(exclude_ids) & ~c["id2"].isin(exclude_ids)
    negatives = c.loc[keep, ["id1", "id2"] + FEATURES].copy()
    negatives["label"] = 0
    return negatives


def training_data(gold_part, exclude_ids):
    positives = gold_part[gold_part["label"] == 1]
    negatives = implied_negatives(positives, exclude_ids)
    data = pd.concat([gold_part[["id1", "id2"] + FEATURES + ["label"]], negatives], ignore_index=True)
    return data.drop_duplicates(["id1", "id2"])


gold_train_val_features = gold_features[gold_features["split"] == "train_val"]
example = implied_negatives(gold_train_val_features[gold_train_val_features["label"] == 1], test_ids)
print(f"{len(example)} implied negatives from the train/validation positives")
print(f"OurAirports types of the implied negatives: {example['id1'].map(oa_idx['type']).value_counts().to_dict()}")

5802 implied negatives from the train/validation positives
OurAirports types of the implied negatives: {'small_airport': 2462, 'heliport': 1674, 'closed': 900, 'medium_airport': 383, 'seaplane_base': 214, 'large_airport': 166, 'balloonport': 3}


### Model selection with 5-fold cross-validation

For each fold, a model is trained on the other four folds (with their implied negatives, leaving out every record of the validation fold and of the test split), scores **all** candidate pairs, and one-to-one matching is applied, exactly as in the final pipeline. Precision, recall and F1 are measured on the gold pairs of the validation fold; `matched_stations` counts the matched stations (always wrong, see section 4) over the whole dataset.

The grid covers the three classifiers, with and without `class_weight="balanced"` (the training data has about ten times more negatives than positives), and five thresholds.

**Selection rule:** the gold F1 alone is not enough, because the gold standard contains no stations: a higher threshold or no class weights gives fewer wrong station matches but a lower gold F1, and vice versa. So among all settings whose CV F1 is within `F1_TOLERANCE = 0.01` of the best one, the setting with the **fewest matched stations** is chosen.

Note on how this rule came about: earlier runs chose the model by gold F1 alone, with a fixed threshold. That chose settings that matched up to four times more stations for less than 0.01 more F1. The rule only uses cross-validation results, but the test results of those runs had been seen.

In [17]:
def make_models(class_weight):
    return {
        "logistic_regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                             LogisticRegression(max_iter=1000, class_weight=class_weight)),
        "random_forest": make_pipeline(SimpleImputer(strategy="median"),
                                       RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                                              class_weight=class_weight, random_state=SEED)),
        "gradient_boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.1,
                                                            class_weight=class_weight, random_state=SEED),
    }


def score_candidates(model):
    scored = candidate_features[["id1", "id2"]].copy()
    scored["score"] = model.predict_proba(candidate_features[FEATURES])[:, 1]
    scored["notes"] = ""
    return scored


CLASS_WEIGHTS = {"none": None, "balanced": "balanced"}
CV_THRESHOLDS = [0.2, 0.3, 0.4, 0.5, 0.6]
cv_rows = []
for fold in range(5):
    train_part = gold_train_val_features[gold_train_val_features["fold"] != fold]
    val_part = gold_train_val_features[gold_train_val_features["fold"] == fold]
    exclude = test_ids | set(val_part["id1"]) | set(val_part["id2"])
    train = training_data(train_part, exclude)
    for weight_name, class_weight in CLASS_WEIGHTS.items():
        for name, model in make_models(class_weight).items():
            scored = score_candidates(model.fit(train[FEATURES], train["label"]))
            for t in CV_THRESHOLDS:
                matched = one_to_one(scored, t)
                cv_rows.append({"model": name, "class_weight": weight_name, "threshold": t, "fold": fold,
                                **evaluate(matched, val_part, t), "matched_stations": matched_stations(matched)})

cv = (pd.DataFrame(cv_rows)
        .groupby(["model", "class_weight", "threshold"])[["precision", "recall", "f1", "matched_stations"]]
        .mean())

F1_TOLERANCE = 0.01
close_to_best = cv[cv["f1"] >= cv["f1"].max() - F1_TOLERANCE].sort_values("matched_stations")
BEST_MODEL, BEST_CLASS_WEIGHT, ML_THRESHOLD = close_to_best.index[0]
print(f"Best CV F1: {cv['f1'].max():.3f}; {len(close_to_best)} settings within {F1_TOLERANCE}")
print(f"Chosen: {BEST_MODEL}, class_weight={BEST_CLASS_WEIGHT}, threshold={ML_THRESHOLD}")
close_to_best.round(3)

Best CV F1: 0.966; 9 settings within 0.01
Chosen: gradient_boosting, class_weight=balanced, threshold=0.6


precision  recall     f1  \
model               class_weight threshold                             
gradient_boosting   balanced     0.6            0.998   0.921  0.958   
random_forest       balanced     0.4            0.997   0.920  0.957   
gradient_boosting   balanced     0.5            0.998   0.921  0.958   
                                 0.4            0.998   0.932  0.964   
random_forest       balanced     0.3            0.997   0.927  0.960   
gradient_boosting   balanced     0.3            0.998   0.930  0.963   
random_forest       balanced     0.2            0.995   0.938  0.966   
gradient_boosting   balanced     0.2            0.998   0.935  0.966   
logistic_regression balanced     0.2            0.995   0.929  0.960   

                                            matched_stations  
model               class_weight threshold                    
gradient_boosting   balanced     0.6                    34.4  
random_forest       balanced     0.4                    42.8  
gradient_boosting   balanced     0.5                    48.2  
                                 0.4                    62.4  
random_forest       balanced     0.3                    64.0  
gradient_boosting   balanced     0.3                    72.0  
random_forest       balanced     0.2                    85.2  
gradient_boosting   balanced     0.2                    87.0  
logistic_regression balanced     0.2                   128.4

All settings, sorted by CV F1 (the trade-off between gold F1 and matched stations):

In [18]:
cv.sort_values("f1", ascending=False).round(3)

precision  recall     f1  \
model               class_weight threshold                             
random_forest       balanced     0.2            0.995   0.938  0.966   
gradient_boosting   balanced     0.2            0.998   0.935  0.966   
                                 0.4            0.998   0.932  0.964   
                                 0.3            0.998   0.930  0.963   
random_forest       balanced     0.3            0.997   0.927  0.960   
logistic_regression balanced     0.2            0.995   0.929  0.960   
gradient_boosting   balanced     0.5            0.998   0.921  0.958   
                                 0.6            0.998   0.921  0.958   
random_forest       balanced     0.4            0.997   0.920  0.957   
gradient_boosting   none         0.2            0.998   0.914  0.954   
random_forest       none         0.2            0.997   0.916  0.954   
logistic_regression balanced     0.3            0.995   0.914  0.952   
random_forest       balanced     0.5            0.997   0.912  0.952   
gradient_boosting   none         0.3            0.998   0.903  0.948   
logistic_regression balanced     0.4            0.995   0.903  0.946   
random_forest       balanced     0.6            0.996   0.901  0.946   
                    none         0.3            0.996   0.896  0.943   
gradient_boosting   none         0.4            0.998   0.890  0.941   
logistic_regression balanced     0.5            0.996   0.888  0.939   
gradient_boosting   none         0.5            0.998   0.882  0.936   
                                 0.6            0.998   0.878  0.934   
logistic_regression balanced     0.6            0.996   0.875  0.931   
random_forest       none         0.4            0.996   0.872  0.930   
                                 0.5            0.996   0.856  0.920   
                                 0.6            0.998   0.836  0.910   
logistic_regression none         0.2            0.996   0.809  0.892   
                                 0.3            0.996   0.778  0.873   
                                 0.4            0.996   0.758  0.860   
                                 0.5            0.996   0.739  0.848   
                                 0.6            0.996   0.690  0.815   

                                            matched_stations  
model               class_weight threshold                    
random_forest       balanced     0.2                    85.2  
gradient_boosting   balanced     0.2                    87.0  
                                 0.4                    62.4  
                                 0.3                    72.0  
random_forest       balanced     0.3                    64.0  
logistic_regression balanced     0.2                   128.4  
gradient_boosting   balanced     0.5                    48.2  
                                 0.6                    34.4  
random_forest       balanced     0.4                    42.8  
gradient_boosting   none         0.2                    24.0  
random_forest       none         0.2                    36.0  
logistic_regression balanced     0.3                   112.8  
random_forest       balanced     0.5                    22.0  
gradient_boosting   none         0.3                    13.6  
logistic_regression balanced     0.4                    95.2  
random_forest       balanced     0.6                    10.0  
                    none         0.3                    20.4  
gradient_boosting   none         0.4                     9.6  
logistic_regression balanced     0.5                    78.6  
gradient_boosting   none         0.5                     7.6  
                                 0.6                     5.4  
logistic_regression balanced     0.6                    67.8  
random_forest       none         0.4                    10.4  
                                 0.5                     3.6  
                                 0.6                     3.0  
logistic_regression none         0.2  

### Train the best model on the full train/validation split and score all candidates

In [19]:
final_training = training_data(gold_train_val_features, test_ids)
print(f"Training data: {int(final_training['label'].sum())} positives, {int((final_training['label'] == 0).sum())} negatives")

classifier = make_models(CLASS_WEIGHTS[BEST_CLASS_WEIGHT])[BEST_MODEL].fit(final_training[FEATURES], final_training["label"])
ml_correspondences = score_candidates(classifier)
print(f"Candidate pairs with probability >= {ML_THRESHOLD}: {int((ml_correspondences['score'] >= ML_THRESHOLD).sum())} "
      f"(for {len(df_kg)} Kaggle records)")

Training data: 617 positives, 6170 negatives


Candidate pairs with probability >= 0.6: 6607 (for 6391 Kaggle records)


## 7. Evaluation on the test split

The test split is used only here. Besides precision, recall and F1 on the gold pairs, the table shows how many matches each matcher produces over the whole dataset and how many of them are stations (always wrong, see section 4).

In [20]:
variants = {
    "rule-based (no 1:1)": (rule_correspondences, RULE_THRESHOLD, False),
    "rule-based + 1:1": (rule_correspondences, RULE_THRESHOLD, True),
    f"ML {BEST_MODEL}, {BEST_CLASS_WEIGHT} (no 1:1)": (ml_correspondences, ML_THRESHOLD, False),
    f"ML {BEST_MODEL}, {BEST_CLASS_WEIGHT} + 1:1": (ml_correspondences, ML_THRESHOLD, True),
}
rows = {}
for label, (correspondences, threshold, with_one_to_one) in variants.items():
    matched = one_to_one(correspondences, threshold) if with_one_to_one else correspondences[correspondences["score"] >= threshold]
    rows[label] = {**evaluate(matched, gold_test, threshold), "threshold": threshold,
                   "matches_in_dataset": len(matched), "matched_stations": matched_stations(matched)}

test_results = pd.DataFrame(rows).T
test_results[["precision", "recall", "f1"]] = test_results[["precision", "recall", "f1"]].astype(float).round(3)
test_results

,precision,recall,f1,true_positives,false_positives,false_negatives,threshold,matches_in_dataset,matched_stations
rule-based (no 1:1),0.733,0.977,0.838,258.0,94.0,6.0,0.4,44214.0,2219.0
rule-based + 1:1,1.000,0.913,0.954,241.0,0.0,23.0,0.4,6284.0,211.0
"ML gradient_boosting, balanced (no 1:1)",0.944,0.955,0.949,252.0,15.0,12.0,0.6,6607.0,131.0
"ML gradient_boosting, balanced + 1:1",1.000,0.924,0.961,244.0,0.0,20.0,0.6,5818.0,35.0


## 8. Error analysis

Results per gold standard group on the test split for the final matcher (ML + one-to-one). For positive groups the share matched is the recall; for negative groups it is the share wrongly matched. `kaggle_matched_to` shows which OurAirports record the Kaggle airport was matched to instead.

In [21]:
final = one_to_one(ml_correspondences, ML_THRESHOLD)
final_set = set(zip(final["id1"], final["id2"]))
kaggle_match = final.set_index("id2")["id1"]

analysis = gold_test.copy()
analysis["matched"] = [p in final_set for p in zip(analysis["id1"], analysis["id2"])]
analysis.groupby("type").agg(pairs=("matched", "size"), matched=("matched", "sum"),
                             share_matched=("matched", "mean")).round(3)

,pairs,matched,share_matched
type,,,
code changed positive,39,39,1.000
hard negative,75,0,0.000
hard positive,75,66,0.880
heliport negative,20,0,0.000
iata positive,75,64,0.853
normal negative,75,0,0.000
normal positive,75,75,1.000


In [22]:
errors = describe_pairs(analysis[analysis["matched"] != (analysis["label"] == 1)])
errors["error"] = np.where(errors["label"] == 1, "false negative", "false positive")
errors["in_candidates"] = [p in candidate_set for p in zip(errors["id1"], errors["id2"])]
errors["ml_score"] = errors[["id1", "id2"]].merge(ml_correspondences, on=["id1", "id2"], how="left")["score"].round(3).values
errors["kaggle_matched_to"] = errors["id2"].map(kaggle_match).map(oa_idx["name"]).fillna("-")
print(f"{len(errors)} errors on the test split")
errors.sort_values(["error", "type"])[["error", "type", "name_oa", "name_kg", "kaggle_matched_to", "city_oa", "city_kg",
                                       "distance_km", "ml_score", "in_candidates"]]

20 errors on the test split


,error,type,name_oa,name_kg,kaggle_matched_to,city_oa,city_kg,distance_km,ml_score,in_candidates
1,false negative,hard positive,tenente lund pressoto airport,franca airport,-,franca,franca,6.2,0.288,True
3,false negative,hard positive,danilson municipal airport,coari airport,-,coari,coari,5.5,0.502,True
6,false negative,hard positive,lille airport,lesquin,-,lesquin,lille,1.1,0.984,True
11,false negative,hard positive,sazena airfield,sazena,-,sazena,lakeside texas,16.5,NaN,False
14,false negative,hard positive,gabes matmata international airport,gabes,-,gabes,gabes,23.3,0.175,True
15,false negative,hard positive,nay pyi taw international airport,naypyitaw,-,naypyitaw,naypyitaw,29.0,0.825,True
18,false negative,hard positive,dunhuang mogao international airport,dunhuang airport,-,dunhuang,dunhuang,29.1,0.539,True
21,false negative,hard positive,zhukovsky international airport,zhukovsky,-,moscow,moscow,2.2,0.988,True
24,false negative,hard positive,dachuan airport,dachuan airport,-,dazhou daxian,daxian,20.0,0.971,True
192,false negative,iata positive,juan jose rondon airport,rondon,rondon airport,paipa,rondon,NaN,0.888,True


**Reading the errors.** All 20 test errors are false negatives (there are no false positives on the test split). They fall into four groups:

- **Not a candidate pair (6).** `sazena` is listed by Kaggle in the US. The other five are `iata positive` pairs without Kaggle coordinates and with differently spelled names (`kokhanok` / `kakhonak`, `tetebedi` / `tetabedi`, `dibba` / `dibaa`, `kamyanets podilskyi` / `kamenets podolskiy`, `kualanamu` / `kuala namu`), see section 2. A fuzzy name blocking rule for records without coordinates could catch them.
- **The OurAirports airport was taken by another Kaggle record during one-to-one matching (5).** Kaggle records with wrong data: `naypyidaw` (`kg_3496`) has the IATA code and city of Eagle Lake, Texas but coordinates in Myanmar, and `hwange national park` (`kg_2486`, IATA `WKM` = Hwange town) takes `hwange national park airport` instead of Kaggle `hwange`. `dazhou jinya airport` (`kg_6175`) takes `dachuan airport`, the old Dazhou airport. `lille` (`kg_5655`, IATA `XDB`) is the Lille-Europe railway station but lies next to the airport, and the station check of section 4 cannot see it because the name has no station word. `ramenskoye` (`kg_2460`) is another name of Zhukovsky airport, so Kaggle contains the airport twice; as decided in the coaching session, such duplicates are left for data fusion.
- **The Kaggle record was matched to a different OurAirports record with the same name (3).** `mount vernon` (no coordinates) is matched to `mount vernon municipal airport` instead of `mount vernon outland airport`: without coordinates the matcher cannot tell apart airports with the same name. `rondon` (no coordinates) is matched to `rondon airport` instead of `juan jose rondon airport`, and `colonia sarmiento` to `colonia sarmiento airport` instead of `lago musters airport`; in both cases the chosen OurAirports record has the same name and no IATA code, so it may be a duplicate or a closely related record in OurAirports rather than a clear matcher error.
- **Score below the threshold (6).** `franca airport`, `coari airport`, `gabes`, `dunhuang airport`, `lake minchumina` and `lewoleba` score between 0.18 and 0.54, below `ML_THRESHOLD = 0.6`. Four of them score above 0.5: the station-based selection rule chose a higher threshold than in earlier runs, which costs these matches but matches fewer stations.

Most remaining errors therefore come from data quality (wrong coordinates or codes, missing coordinates, duplicates, unlabelled stations) rather than from the matcher.

## 9. Save the correspondences

The final correspondences (ML matcher + one-to-one matching, threshold `ML_THRESHOLD`) over all candidate pairs are saved for the data fusion step.

In [23]:
output_file = OUTPUT_DIR / "correspondences_oa_kg.csv"
final[["id1", "id2", "score"]].sort_values("score", ascending=False).to_csv(output_file, index=False)

print(f"Saved {len(final)} correspondences to {output_file}")
print(f"Kaggle records matched to an OurAirports record: {final['id2'].nunique()} of {len(df_kg)} "
      f"({final['id2'].nunique() / len(df_kg):.1%})")
print(f"OurAirports types of the matches: {final['id1'].map(oa_idx['type']).value_counts().to_dict()}")

Saved 5818 correspondences to ../../../data/identity_resolution/outputs/correspondences_oa_kg.csv
Kaggle records matched to an OurAirports record: 5818 of 6391 (91.0%)
OurAirports types of the matches: {'medium_airport': 2886, 'small_airport': 1672, 'large_airport': 1110, 'seaplane_base': 74, 'heliport': 38, 'closed': 38}
